# 🌾 KrishiAI — Convert Pre-trained PlantVillage Model to TFLite

**No training needed! Takes ~5 minutes.**

- Source model: `linkanjarad/mobilenet_v2_1.0_224-plant-disease-identification`
- Accuracy: **95.4%** on PlantVillage (38 classes)
- Output: `crop_disease.tflite` + `class_names.json`
- Size: ~9MB → ~3MB after INT8 quantization

## Steps
1. Install dependencies
2. Download pre-trained model from Hugging Face
3. Map 38 PlantVillage classes → 13 KrishiAI disease keys
4. Convert to TFLite INT8
5. Download both files

In [ ]:
# Step 1: Install
!pip install -q transformers torch pillow numpy tensorflow

In [ ]:
# Step 2: Download model + get class labels
from transformers import AutoImageProcessor, AutoModelForImageClassification
import json

MODEL_ID = 'linkanjarad/mobilenet_v2_1.0_224-plant-disease-identification'

print('Downloading model...')
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageClassification.from_pretrained(MODEL_ID)
model.eval()

# Get all 38 PlantVillage class labels
id2label = model.config.id2label
print(f'\nModel has {len(id2label)} classes:')
for i, label in sorted(id2label.items(), key=lambda x: int(x[0])):
    print(f'  {i:2d}: {label}')

In [ ]:
# Step 3: Map 38 PlantVillage classes → 13 KrishiAI disease keys
# We keep ALL 38 outputs from the model but map them to our 13 labels

KRISHIAI_MAP = {}
for idx, label in id2label.items():
    label_lower = label.lower()

    # Rice
    if 'rice' in label_lower and 'blast' in label_lower:
        KRISHIAI_MAP[int(idx)] = 'rice_blast'
    elif 'rice' in label_lower and ('blight' in label_lower or 'bacterial' in label_lower):
        KRISHIAI_MAP[int(idx)] = 'rice_blight'
    elif 'rice' in label_lower and 'brown_spot' in label_lower:
        KRISHIAI_MAP[int(idx)] = 'rice_brown_spot'
    elif 'rice' in label_lower and 'hispa' in label_lower:
        KRISHIAI_MAP[int(idx)] = 'rice_stem_borer'
    elif 'rice' in label_lower and 'healthy' in label_lower:
        KRISHIAI_MAP[int(idx)] = 'healthy'
    # Wheat
    elif 'wheat' in label_lower and ('rust' in label_lower or 'scab' in label_lower):
        KRISHIAI_MAP[int(idx)] = 'wheat_rust'
    elif 'wheat' in label_lower and 'healthy' in label_lower:
        KRISHIAI_MAP[int(idx)] = 'healthy'
    # Potato
    elif 'potato' in label_lower and 'late_blight' in label_lower:
        KRISHIAI_MAP[int(idx)] = 'potato_late_blight'
    elif 'potato' in label_lower and 'early_blight' in label_lower:
        KRISHIAI_MAP[int(idx)] = 'potato_late_blight'
    elif 'potato' in label_lower and 'healthy' in label_lower:
        KRISHIAI_MAP[int(idx)] = 'healthy'
    # Tomato
    elif 'tomato' in label_lower and ('yellow_leaf_curl' in label_lower or 'ylcv' in label_lower):
        KRISHIAI_MAP[int(idx)] = 'tomato_leaf_curl'
    elif 'tomato' in label_lower and 'early_blight' in label_lower:
        KRISHIAI_MAP[int(idx)] = 'tomato_blight'
    elif 'tomato' in label_lower and 'late_blight' in label_lower:
        KRISHIAI_MAP[int(idx)] = 'tomato_blight'
    elif 'tomato' in label_lower and 'healthy' in label_lower:
        KRISHIAI_MAP[int(idx)] = 'healthy'
    # Pepper as proxy for brinjal
    elif 'pepper' in label_lower and 'spot' in label_lower:
        KRISHIAI_MAP[int(idx)] = 'brinjal_borer'
    elif 'pepper' in label_lower and 'healthy' in label_lower:
        KRISHIAI_MAP[int(idx)] = 'healthy'
    # Corn as proxy for nitrogen deficiency
    elif 'corn' in label_lower or 'maize' in label_lower:
        if 'blight' in label_lower or 'spot' in label_lower:
            KRISHIAI_MAP[int(idx)] = 'nitrogen_deficiency'
        else:
            KRISHIAI_MAP[int(idx)] = 'healthy'
    # All other healthy
    elif 'healthy' in label_lower:
        KRISHIAI_MAP[int(idx)] = 'healthy'
    # Everything else maps to unknown (will trigger cloud vision fallback)
    else:
        KRISHIAI_MAP[int(idx)] = 'unknown'

print('PlantVillage → KrishiAI mapping:')
for idx, label in sorted(id2label.items(), key=lambda x: int(x[0])):
    mapped = KRISHIAI_MAP.get(int(idx), 'unknown')
    print(f'  {int(idx):2d}: {label:50s} → {mapped}')

# Save mapping
with open('/content/label_map.json', 'w') as f:
    json.dump({str(k): v for k, v in KRISHIAI_MAP.items()}, f, indent=2)
print('\n✅ label_map.json saved')

In [ ]:
# Step 4: Export to ONNX → TFLite
# (PyTorch → ONNX → TF → TFLite is most reliable path)
!pip install -q onnx onnx2tf

import torch
import numpy as np

# Export to ONNX first
dummy_input = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    model,
    dummy_input,
    '/content/crop_disease.onnx',
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
    opset_version=11,
)
print('✅ ONNX exported')

In [ ]:
# Convert ONNX → TFLite via onnx2tf
!onnx2tf -i /content/crop_disease.onnx -o /content/tf_model -oiqt
print('✅ TFLite conversion done')

In [ ]:
# Find the generated tflite file
import os, glob
tflite_files = glob.glob('/content/tf_model/**/*.tflite', recursive=True)
print('Generated TFLite files:', tflite_files)

# Pick the INT8 quantized version (smallest)
tflite_files.sort(key=os.path.getsize)
best = tflite_files[0]
size_mb = os.path.getsize(best) / (1024*1024)
print(f'Using: {best} ({size_mb:.1f} MB)')

import shutil
shutil.copy(best, '/content/crop_disease.tflite')
print('✅ Copied to /content/crop_disease.tflite')

In [ ]:
# Step 5: Save class_names.json (38 original PlantVillage labels in order)
import json

# Full 38-class label list in index order
class_names_38 = [id2label[str(i)] for i in range(len(id2label))]

# Also save the krishiai mapping for the app
label_map = {str(i): KRISHIAI_MAP.get(i, 'unknown') for i in range(len(id2label))}

with open('/content/class_names.json', 'w') as f:
    json.dump(class_names_38, f, indent=2)

with open('/content/label_map.json', 'w') as f:
    json.dump(label_map, f, indent=2)

print('✅ class_names.json (38 labels)')
print('✅ label_map.json (PlantVillage → KrishiAI mapping)')
print()
print('Labels:', class_names_38)

In [ ]:
# Step 6: Quick verify
import tensorflow as tf
import numpy as np

interp = tf.lite.Interpreter('/content/crop_disease.tflite')
interp.allocate_tensors()
inp = interp.get_input_details()
out = interp.get_output_details()
print('Input :', inp[0]['shape'], inp[0]['dtype'])
print('Output:', out[0]['shape'], out[0]['dtype'])

# Test with random image
test_img = np.random.randint(0, 255, inp[0]['shape'], dtype=np.uint8 if inp[0]['dtype']==np.uint8 else np.float32)
interp.set_tensor(inp[0]['index'], test_img)
interp.invoke()
result = interp.get_tensor(out[0]['index'])
print('Output shape:', result.shape, '✅ Model works!')

In [ ]:
# Step 7: Download all 3 files
from google.colab import files

print('Downloading files...')
files.download('/content/crop_disease.tflite')  # → assets/models/
files.download('/content/class_names.json')     # → assets/models/
files.download('/content/label_map.json')       # → assets/models/

import os
size = os.path.getsize('/content/crop_disease.tflite') / (1024*1024)
print(f'\n✅ Done! crop_disease.tflite = {size:.1f} MB')
print()
print('Next steps:')
print('  1. Copy all 3 files to app.krishiai/assets/models/')
print('  2. Share them here — Claude will integrate into the app automatically')